# Laboratorio 02. Neurona y perceptrón

**Pregunta central:** ¿Qué calcula una neurona y qué tipo de frontera puede aprender un perceptrón?

## Objetivos

- Calcular una combinación ponderada, sesgo y activación.
- Implementar la actualización del perceptrón en NumPy.
- Contrastar AND separable con XOR no separable por una recta.

## Prerrequisitos y conexión

Conecta la forma tensorial del laboratorio 01 con parámetros interpretables. Se requieren productos matriciales y comparación de etiquetas binarias.

## Teoría breve

Una neurona calcula $z=w^Tx+b$ y $a=\phi(z)$. El perceptrón actualiza sus parámetros cuando una observación se clasifica incorrectamente; una única frontera lineal no puede separar XOR.

## Problema y datos

Usamos las cuatro combinaciones binarias de AND y XOR, enumeradas por completo y sin descarga externa.

## Evaluación

La exactitud sobre la tabla exhaustiva comprueba la regla de AND; la gráfica de XOR evidencia por qué una frontera lineal no basta. Esta tabla no estima generalización.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Cálculo explícito en NumPy
Computamos z y la sigmoide paso a paso para tres observaciones.


In [ ]:
X = np.array([[0.2, 0.8], [0.7, 0.1], [0.5, 0.6]], dtype=np.float32)
w = np.array([1.2, -0.7], dtype=np.float32)
b = -0.1
z = X @ w + b
activation = 1 / (1 + np.exp(-z))
print('z:', z.round(3))
print('sigmoid(z):', activation.round(3))


**Resultado e interpretación.** La misma regla se aplica fila por fila. La sigmoide lleva los valores a (0, 1), pero no convierte por sí sola una probabilidad en una decisión calibrada.


### Equivalencia con PyTorch
Copiamos los mismos parámetros y calculamos gradientes respecto a los pesos.


In [ ]:
linear = nn.Linear(2, 1)
with torch.no_grad():
    linear.weight.copy_(torch.tensor(w).reshape(1, 2))
    linear.bias.fill_(b)
torch_output = torch.sigmoid(linear(torch.tensor(X)))
print('salidas PyTorch:', torch_output.detach().squeeze().numpy().round(3))
torch_output.sum().backward()
print('gradiente de pesos:', linear.weight.grad)


**Resultado e interpretación.** Los valores coinciden salvo redondeo. Autograd calcula derivadas de la salida respecto a cada parámetro, mecanismo que se usará para aprenderlos.


### Entrenar AND desde cero
Usamos la tabla de verdad como conjunto de entrenamiento y calculamos la exactitud.


In [ ]:
X = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
y = np.array([0, 0, 0, 1])
w = np.zeros(2); b = 0.0; rate = 0.2
for epoch in range(12):
    for row, target in zip(X, y):
        predicted = int(row @ w + b > 0)
        error = target - predicted
        w += rate * error * row
        b += rate * error
predictions = (X @ w + b > 0).astype(int)
print('w, b:', w, b, '| predicciones:', predictions, '| exactitud:', (predictions == y).mean())


**Resultado e interpretación.** La regla modifica los parámetros solo cuando hay error. AND es un problema pequeño; esta exactitud no mide generalización a una población nueva.


### Contrastar XOR
Inspeccionamos las dos clases y trazamos la recta que el perceptrón encontró para AND.


In [ ]:
xor = np.array([0, 1, 1, 0])
print('XOR:', xor, '| etiquetas predichas por la frontera AND:', predictions)
xx = np.linspace(-0.2, 1.2, 100)
if abs(w[1]) > 1e-8:
    yy = -(w[0] * xx + b) / w[1]
    plt.scatter(X[:, 0], X[:, 1], c=xor, cmap='coolwarm', s=100)
    plt.plot(xx, yy, color='black'); plt.xlim(-0.2, 1.2); plt.ylim(-0.2, 1.2)
    plt.xlabel('x1'); plt.ylabel('x2'); plt.title('XOR y frontera lineal'); plt.show()


**Resultado e interpretación.** Las etiquetas XOR alternan en las esquinas, por lo que ninguna recta puede separarlas perfectamente. Una arquitectura con capa oculta será necesaria.


## Limitaciones y conclusiones

El perceptrón define una frontera lineal y depende del orden de actualización. La tabla AND/XOR es exhaustiva para las cuatro entradas binarias, por lo que verifica una propiedad geométrica pero no estima generalización a nuevas observaciones.

## Ejercicios

1. **Guiado:** calcula a mano una actualización del perceptrón para una fila mal clasificada.
2. **Independiente:** cambia el orden de las filas de AND y registra cuántas épocas necesita.
3. **Desafío:** propone una representación que vuelva separable XOR y verifica la hipótesis.

**Siguiente:** en el MLP NumPy construiremos una capa oculta y una representación no lineal.